---
# COSC2673 | COSC2793 (Computational) Machine Learning
## Week 6 Lab: **Decision Trees**
---

# Introduction

In the last few weeks, we covered the typical machine learning model development process. In this week's lab, we will explore decision tree-based models.

This lab assumes you have completed the prior labs. If not, please complete them first.

## Objectives
- Continue becoming familiar with Python and machine learning packages.
- Learn classification decision trees using both categorical and continuous numerical data.
- Compare the performance of different trees, including trees after pruning.
- Learn regression decision trees and compare them to regression models from previous labs.

## Dataset

The data is related to direct marketing campaigns of a Portuguese banking institution. The marketing campaigns were based on phone calls. Often, more than one contact to the same client was required in order to determine whether the product (bank term deposit) would be subscribed.

#### Input variables:
- Bank client data:
    1. age (numeric)
    2. job : type of job (categorical: "admin.","unknown","unemployed","management","housemaid","entrepreneur","student", "blue-collar","self-employed","retired","technician","services") 
    3. marital : marital status (categorical: "married","divorced","single"; note: "divorced" means divorced or widowed)
    4. education (categorical: "unknown","secondary","primary","tertiary")
    5. default: has credit in default? (binary: "yes","no")
    6. balance: average yearly balance, in euros (numeric) 
    7. housing: has housing loan? (binary: "yes","no")
    8. loan: has personal loan? (binary: "yes","no")
- Related with the last contact of the current campaign:
    9. contact: contact communication type (categorical: "unknown","telephone","cellular") 
    10. day: last contact day of the month (numeric)
    11. month: last contact month of year (categorical: "jan", "feb", "mar", ..., "nov", "dec")
    12. duration: last contact duration, in seconds (numeric)
- Other attributes:
    13. campaign: number of contacts performed during this campaign and for this client (numeric, includes last contact)
    14. pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric, -1 means client was not previously contacted)
    15. previous: number of contacts performed before this campaign and for this client (numeric)
    16. poutcome: outcome of the previous marketing campaign (categorical: "unknown","other","failure","success")
    
#### Output variable (desired target):   
    17. y - has the client subscribed a term deposit? (binary: "yes","no")
    
This dataset is publicly available for research. For details, see Moro et al. (2011).

Moro et al., 2011: S. Moro, R. Laureano and P. Cortez. Using Data Mining for Bank Direct Marketing: An Application of the CRISP-DM Methodology. In P. Novais et al. (Eds.), Proceedings of the European Simulation and Modelling Conference - ESM'2011, pp. 117-121, Guimarães, Portugal, October, 2011. 

Let's read the data first.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

data = pd.read_csv('./bank-full.csv', delimiter=';')
data.head()

The dataset contains both categorical and numerical attributes. Let's convert the categorical columns to the pandas categorical data type.


In [ ]:
for col in data.columns:
    if data[col].dtype == object:
        data[col] = data[col].astype('category')

Scikit-learn's decision tree classifier does not accept categorical input features. It requires numeric feature values, while the target labels can be categorical. Therefore, we must convert the categorical attributes into a suitable numeric representation. Pandas can help with this.

First, split the dataset into the target variable and the feature attributes:


In [ ]:
dataY = data['y']
dataX = data.drop(columns='y')

Then use pandas to generate numeric versions of the categorical attributes:

In [ ]:
dataXExpand = pd.get_dummies(dataX)
dataXExpand.head()

As you can see, the categories are expanded into indicator variables with values 0 or 1. This representation allows the decision tree to handle categorical features as numeric data. It is not ideal, but it enables learning a valid decision tree.

Discuss the following with your tutor:
- Why is it better to convert categorical attributes into binary indicator variables than to assign arbitrary integer codes?
- What problems can arise if categories are encoded as integers?

The target labels also need preprocessing. Scikit-learn can handle categorical targets, but it expects them to be encoded as integers rather than strings. We use sklearn.preprocessing.LabelEncoder to convert strings to numeric labels. The print statement below shows the mapping from original string labels to integer codes.

In [ ]:
from sklearn import preprocessing

le = preprocessing.LabelEncoder()
le.fit(dataY)
class_labels = le.classes_
dataY = le.transform(dataY)

print("Class label mapping:", list(class_labels))

# Exploratory Data Analysis

Task: Since EDA has been covered in previous labs, this section is left as an exercise. Perform exploratory data analysis and use your findings to justify the modeling decisions in the following code blocks.

In [ ]:
# TODO

# Setting up the performance metric
Several metrics are appropriate for this problem, such as `accuracy_score`, `f1_score`, and others. More information is available in sklearn's metrics documentation: https://scikit-learn.org/stable/modules/classes.html#module-sklearn.metrics

The insights gained from EDA are important when choosing the right metric. Use your EDA results to identify the characteristics that matter most, and select the best performance measure accordingly. Discuss your choice with your tutor.

For this task, we want to treat all classes equally, so we will use the macro-averaged `f1_score` and aim for a target value of 75%. 

Note that F1-score is not the only valid metric for this problem.

# Setup the experiment - data splits

**Which data should we use for evaluation?**

We can simulate unseen data in several ways:
1. Hold-out validation
2. Cross-validation

For this experiment, we will use hold-out validation.

Task: Use what you learned in the few of weeks to split the data appropriately.

In [ ]:
from sklearn.model_selection import train_test_split

train_val_X, test_data_X, train_val_y, test_data_y = train_test_split(
    dataXExpand, dataY, test_size=0.2, shuffle=True, random_state=0
)

train_data_X, val_data_X, train_data_y, val_data_y = train_test_split(
    train_val_X, train_val_y, test_size=0.25, shuffle=True, random_state=0
)

print(train_data_X.shape, val_data_X.shape, test_data_X.shape)

In [ ]:
train_X = train_data_X.to_numpy()
train_y = train_data_y

test_X = test_data_X.to_numpy()
test_y = test_data_y

val_X = val_data_X.to_numpy()
val_y = val_data_y

In [ ]:
from sklearn.metrics import f1_score

def get_acc_scores(clf, train_X, train_y, val_X, val_y):
    train_pred = clf.predict(train_X)
    val_pred = clf.predict(val_X)
    
    train_acc = f1_score(train_y, train_pred, average='macro')
    val_acc = f1_score(val_y, val_pred, average='macro')
    
    return train_acc, val_acc

# Simple decision tree training

Let's train a simple decision tree and visualize it.


In [ ]:
from sklearn import tree

tree_max_depth = 2   # change this value and observe

clf = tree.DecisionTreeClassifier(criterion='entropy', max_depth=tree_max_depth, class_weight='balanced')
clf = clf.fit(train_X, train_y)

In [ ]:
tree.plot_tree(clf)

In [ ]:
# A nicer visualization can be produced using the graphviz package. 
# This is beyond the scope of this lab but the following code can be used to produce a visualization using graphviz.

# Install python-graphviz in conda with the following command (or similar for different setups).
# conda install -c conda-forge python-graphviz

# import graphviz 

# def get_tree_2_plot(clf):
#     dot_data = tree.export_graphviz(clf, out_file=None, 
#                       feature_names=dataXExpand.columns,  
#                       class_names=class_labels,  
#                       filled=True, rounded=True,  
#                       special_characters=True)  
#     graph = graphviz.Source(dot_data) 
#     return graph

# Dtree = get_tree_2_plot(clf)
# Dtree

In [ ]:
train_acc, val_acc = get_acc_scores(clf,train_X, train_y, val_X, val_y)
print("Train f1 score: {:.3f}".format(train_acc))
print("Validation f1 score: {:.3f}".format(val_acc))

Did we achieve the desired target value? If not, what do you think the above results indicate: overfitting or underfitting?

Based on the answer to the above question, what do you think is the best course of action?


## Hyperparameter tuning

What are the important hyperparameters of `DecisionTreeClassifier`?

Tune the key hyperparameters identified above to improve performance. In this example, we explore `max_depth` and `min_samples_split`.

We will use `GridSearchCV` to perform cross-validated hyperparameter tuning. This step may take some time depending on your computer.

In [ ]:
from sklearn.model_selection import GridSearchCV

parameters = {'max_depth': np.arange(2, 400, 50), 'min_samples_split': np.arange(2, 50, 5)}

dt_clf = tree.DecisionTreeClassifier(criterion='entropy', class_weight='balanced')
grid_clf = GridSearchCV(dt_clf, parameters, scoring='f1_macro')
grid_clf.fit(train_X, train_y)

In [ ]:
pd.DataFrame(grid_clf.cv_results_)

In [ ]:
print(grid_clf.best_score_)
print(grid_clf.best_params_)

clf = grid_clf.best_estimator_

In [ ]:
train_acc, val_acc = get_acc_scores(clf,train_X, train_y, val_X, val_y)
print("Train f1 score: {:.3f}".format(train_acc))
print("Validation f1 score: {:.3f}".format(val_acc))

Did we achieve the desired target value? If not, what do you think the above results indicate: overfitting or underfitting?

Based on the answer to the above question, what do you think is the best course of action?

## Post-pruning decision trees with cost complexity pruning

DecisionTreeClassifier includes parameters such as `min_samples_leaf` and `max_depth` to limit tree growth and prevent overfitting. These are pre-pruning controls.

Minimal cost-complexity pruning is a post-pruning technique that prunes a fully grown tree based on an effective alpha value. The algorithm removes the weakest branches first, using the smallest alpha values to decide which nodes to prune.


In [ ]:
clf = tree.DecisionTreeClassifier(class_weight='balanced')
path = clf.cost_complexity_pruning_path(train_X, train_y)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

*This step may take a while depending on your computer.*

In [ ]:
clfs = []
for ccp_alpha in ccp_alphas:
    clf = tree.DecisionTreeClassifier(random_state=0, ccp_alpha=ccp_alpha, class_weight='balanced')
    clf.fit(train_X, train_y)
    clfs.append(clf)

In [ ]:
train_scores = [f1_score(train_y, clf.predict(train_X), average='macro') for clf in clfs]
val_scores = [f1_score(val_y, clf.predict(val_X), average='macro') for clf in clfs]

fig, ax = plt.subplots(figsize=(10,10))
ax.set_xlabel("alpha")
ax.set_ylabel("f1_score")
ax.set_title("Accuracy vs alpha for training and validation sets")
ax.plot(ccp_alphas, train_scores, marker='o', label="train",
        drawstyle="steps-post")
ax.plot(ccp_alphas, val_scores, marker='o', label="validation",
        drawstyle="steps-post")
ax.legend()
plt.show()

What `ccp_alpha` value would you choose as the best for the task?

# Random forest

Random forests build many trees using the same dataset. A single decision tree run repeatedly on the same data will produce the same result, so random forests introduce randomness by training each tree on different random subsets of samples and features. This uses bootstrapped datasets to create diverse trees.

Scikit-learn's `RandomForestClassifier` handles this automatically. Let's apply it to our dataset.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
clf = RandomForestClassifier(max_depth=8, n_estimators=500, class_weight='balanced_subsample', random_state=0)
clf.fit(train_X, train_y)

In [ ]:
train_acc, val_acc = get_acc_scores(clf, train_X, train_y, val_X, val_y)
print("Train f1 score: {:.3f}".format(train_acc))
print("Validation f1 score: {:.3f}".format(val_acc))

Task: Now tune the hyper parameters of the random forest.

Is the final model after hyperparameter tuning better than the previous decision tree model? Why or why not?

Let's visualise the feature importance of the random forest classifier.

In [ ]:
tree_feature_importances = clf.feature_importances_
sorted_idx = tree_feature_importances.argsort()

plt.figure(figsize=(10,10))
plt.barh(dataXExpand.columns[sorted_idx], tree_feature_importances[sorted_idx])
plt.show()

Based on the figure above, do you see any reason for concern about the model?

If the model relies heavily on `duration` to predict the target, what issue could that cause?

# Exercise: Regression Decision Tree

A regression decision tree can also be trained. These trees predict a continuous value at each leaf node. You will investigate regression trees using the Boston housing dataset from previous labs.

The code snippet below will help you get started. Note that entropy is not used for regression splits, so the default criterion in sklearn is appropriate. DecisionTreeRegressor also supports similar pre-pruning parameters.

In [ ]:
# import pandas as pd
# import matplotlib.pyplot as plt
# import numpy as np
# import sklearn 

# from sklearn import tree
# from sklearn import preprocessing
# from sklearn import metrics
# from sklearn import model_selection

# Load dataset

# bostonDataTarget = bostonData['MEDV']
# bostonDataAttrs = bostonData.drop(columns='MEDV')
# trainY, testY, trainX, testX = model_selection.train_test_split(np.array(bostonDataTarget),np.array(bostonDataAttrs), test_size=0.2)
# clfBoston = sklearn.tree.DecisionTreeRegressor(max_depth=5, min_samples_split=5)
# clfBoston = clfBoston.fit(trainX, trainY)
# predictions = clfBoston.predict(testX)
# metrics.mean_squared_error(testY, predictions)

How does the error of the regression decision tree compare to the best results from previous labs?

Find a good set of pre-pruning parameters that minimize the mean squared error.